### STRUCTURED OUPTUT

Models can be requestde to proevide their response in a format matching a given schema. This is usefil for ensuring thhe output can be easily parsed and used in a subsequent processing. LangChain suooirts multiplr schema types and methods for enforcing structured output.

### Pydantic

Pydantic mdeols provied the richest feature set with field validation, descriptions, and nested structures.

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x110a766f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x110e2acc0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year in which the movie was released")
    director:str=Field(description="Name of the director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x110a766f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x110e2acc0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year in which the movie was released', 'type': 'integer'}, 'director': {'description': 'Name of the director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating o

In [7]:
model.invoke("Provides details about the movie The Lord of the Rings: The Fellowship of the Ring")

AIMessage(content='<think>\nOkay, so I need to provide details about "The Lord of the Rings: The Fellowship of the Ring." Let me start by recalling what I know. First, it\'s the first movie in the Lord of the Rings trilogy directed by Peter Jackson, right? It\'s based on J.R.R. Tolkien\'s book. The main characters include Frodo Baggins, who has the Ring, and he\'s trying to get rid of it. There\'s a lot of other characters like Gandalf, Aragorn, Legolas, Gimli, and the rest of the Fellowship.\n\nThe movie was released in 2001, I think. The production was big, with a lot of special effects, especially for the time. It won several Academy Awards, maybe for Best Cinematography or Best Visual Effects. The score was composed by Howard Shore, and it\'s famous for the leitmotifs, like the Shire theme and the Ring theme.\n\nI should mention the plot: Frodo and his friends must travel to Mordor to destroy the One Ring in the fires where it was made. They face many challenges, like the attack by

In [9]:
response = model_with_structure.invoke("Provides details about the movie The Lord of the Rings: The Fellowship of the Ring")
response

Movie(title='The Lord of the Rings: The Fellowship of the Ring', year=2001, director='Peter Jackson', rating=8.8)

### MESSAGE OUTPUT ALONGSIDE PARSED STRUCTURE

In [13]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A Movie with details."""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The year in which the movie was released")
    director:str=Field(...,description="Name of the director of the movie")
    rating:float=Field(...,description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provides details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what functions I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception.\n\nFirst, the title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb does list it at 8.8. So I have all the required parameters. I\'ll structure the tool call with these details. Make sure the JSON is correctly formatted with the right data types: year as integer, rating as a number. No missing fields. That should cover it.\n', 'tool_calls': [{'id': 'pvhbpve6g', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'c

### NESTED STRUCTURE

In [15]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetail(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in Millions USD")

model_with_structure = model.with_structured_output(MovieDetail)

response = model_with_structure.invoke("Tell me about The Amazing Spider-Man.")
response



MovieDetail(title='The Amazing Spider-Man', year=2012, cast=[Actor(name='Andrew Garfield', role='Peter Parker / Spider-Man'), Actor(name='Emma Stone', role='Gwen Stacy'), Actor(name='Rhys Ifans', role='Dr. Curt Connors / The Lizard')], genres=['Action', 'Superhero', 'Adventure'], budget=130.0)

### TYPEDDICT

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [21]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[str, ..., "The movie's rating out of 10"]

model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("Tell me about the The Amazing Spider-Man 2.")
response

{'director': 'Marc Webb',
 'rating': '7.0',
 'title': 'The Amazing Spider-Man 2',
 'year': 2014}

In [20]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetail(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in Millions USD")

model_with_structure = model.with_structured_output(MovieDetail)

response = model_with_structure.invoke("Tell me about The Amazing Spider-Man.")
response



{'budget': 220000000,
 'cast': [{'name': 'Andrew Garfield', 'role': 'Peter Parker / Spider-Man'},
  {'name': 'Emma Stone', 'role': 'Gwen Stacy'},
  {'name': 'Jamie Foxx', 'role': 'Max Dillon / Electro'},
  {'name': 'Sally Field', 'role': 'Rhino (voice)'},
  {'name': 'Martin Sheen',
   'role': 'Ben Reilly / Scarlet Spider (motion capture)'}],
 'genres': ['Action', 'Superhero', 'Adventure'],
 'title': 'The Amazing Spider-Man',
 'year': 2012}

In [23]:
model.profile


{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DATACLASSES

A dataclass is a class typically containing mainly data, although there aren't really any restriciton. You can create it using the @dataclass decorator

In [ ]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ")

In [35]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact Information for a person"""
    name: str = Field(description="The Name of the person")
    email: str = Field(description="The email of the person")
    phone: int = Field(description="The phone number of the person")

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo #Auto Select Providing Strategy
)

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract the contact information from: John Cena, johncenacantseeme@gmail.com, (242) 124-4134"}]
})

result 

{'messages': [HumanMessage(content='Extract the contact information from: John Cena, johncenacantseeme@gmail.com, (242) 124-4134', additional_kwargs={}, response_metadata={}, id='273b9549-d06b-46b1-9630-c0bfb70db9d9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "John Cena, johncenacantseeme@gmail.com, (242) 124-4134". \n\nFirst, I need to identify the different parts. The name is clearly "John Cena". Then there\'s the email address, which looks like "johncenacantseeme@gmail.com". The phone number is "(242) 124-4134". \n\nWait, the phone number has parentheses and a space. The function parameters expect an integer for the phone number. So I should remove the parentheses and dashes to make it a straight number. Let me check the format. The function\'s phone parameter is type integer, so it should be 2421244134 without any symbols.\n\nNow, I need to structure this into the Contact

In [34]:
print(result["structured_response"])
#ContactInfo(name='John Cena', email='johncenacantseeme@gmail.com', phone='(242) 124-4134)

name='John Cena' email='johncenacantseeme@gmail.com' phone=2421244134


In [40]:
## TypedDict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """COntact Information for a person"""
    name: str #The name of the person
    email: str #The email of the person
    phone: int #Phone no of the person

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo #Auto-selct providerStrategy
)

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract the contact information from: John Cena, johncenacantseeme@gmail.com, (242) 124-4134"}]
})

result["structured_response"]

{'name': 'John Cena',
 'email': 'johncenacantseeme@gmail.com',
 'phone': 2421244134}

In [41]:
## DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a perosn"""
    name:str #The name of the person
    email:str #The email of the perosn
    phone:str #The phone no of the person

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract the contact information from: John Cena, johncenacantseeme@gmail.com, (242) 124-4134"}]
})

result["structured_response"]



ContactInfo(name='John Cena', email='johncenacantseeme@gmail.com', phone='(242) 124-4134')